# B (left, middle): weighted slope and axis ratio per scenario

Fills layer `slope_axis_ratio` of the figure SVG.

In [1]:
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Helvetica'] + matplotlib.rcParams['font.sans-serif']
matplotlib.rcParams['font.size'] = 6
matplotlib.rcParams['text.usetex'] = False
matplotlib.rcParams["ps.usedistiller"] = 'xpdf'
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.weight'] = 'normal'
matplotlib.rcParams["mathtext.fontset"] = 'cm'

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import copy

import figurefirst as fifi
import fly_plot_lib.plot as fpl

from align_course_direction_analysis import unifying_algo_analysis as uaa
from splitflow.set_zorder_functions import *

from scenarios import FIGURE_NAME, SCENARIOS, LOW_SCENARIOS, HIGH_SCENARIOS, SHORT_LABELS, COLORS, get_unifying_filename, save_layer

/Users/florisvb/src/SplitFlowOptoTracking/.venv/lib/python3.11/site-packages/figurefirst/mpl_fig_to_figurefirst_svg.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Using device: cpu


In [3]:
MARKERSIZE = 0.75
SLOPE_YLIM = (0, 8)        # low-range axis, same as the fly figures
SLOPE_HIGH_YLIM = (0, 30)  # Brownian and run and tumble
fifi_figure_label = 'slope_axis_ratio'

In [4]:
def set_scenario_xticks(ax, keys, show_labels=True):
    ax.set_xticks(np.arange(1, len(keys) + 1))
    if show_labels:
        ax.set_xticklabels([SHORT_LABELS[key] for key in keys], rotation=45, ha='right', rotation_mode='anchor')
    else:
        ax.set_xticklabels([])

In [5]:
def get_median_slope_and_axis_ratio(fname):
    df = pd.read_parquet(fname)

    slopes = []
    axis_ratios = []
    rotation = []

    for obj_id in df.obj_id_unique_event.unique():
        data = df[df.obj_id_unique_event==obj_id].copy()

        data['slope_abs'] = data.slope.abs()
        slopes.append( uaa.get_weighted_value(data, col='slope_abs', use='rmse_affine') )

        axis_ratios.append( uaa.get_weighted_value(data, col='axis_ratio', use='rmse_affine') )

        data.rotation = data.rotation.abs()
        rotation.append( uaa.get_weighted_value(data, col='rotation', use='rmse_affine') )

    results = {'slopes': np.array(slopes)/0.01,
               'axis_ratios': np.array(axis_ratios),
               'rotation': np.array(rotation),
              }
    return results

In [6]:
results = {key: get_median_slope_and_axis_ratio(get_unifying_filename(key)) for key in SCENARIOS}
for key in SCENARIOS:
    r = results[key]
    ylim = SLOPE_HIGH_YLIM if key in HIGH_SCENARIOS else SLOPE_YLIM
    print(key, 'n =', len(r['slopes']),
          ' median slope = %.2f' % np.median(r['slopes']),
          ' median axis ratio = %.2f' % np.median(r['axis_ratios']),
          ' slopes above ylim (%g):' % ylim[1], np.sum(r['slopes'] > ylim[1]))

shuffled_laminar n = 200  median slope = 1.03  median axis ratio = 0.53  slopes above ylim (8): 0
shuffled_stillair n = 200  median slope = 1.34  median axis ratio = 0.54  slopes above ylim (8): 3
levy_flight n = 200  median slope = 0.03  median axis ratio = 0.15  slopes above ylim (8): 0
random_turns n = 200  median slope = 5.02  median axis ratio = 0.47  slopes above ylim (8): 10
brownian n = 200  median slope = 10.54  median axis ratio = 0.38  slopes above ylim (30): 0
run_and_tumble n = 200  median slope = 9.66  median axis ratio = 0.37  slopes above ylim (30): 0


In [7]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

In [8]:
def plot_feature(ax, keys, feature, ylim, yticks, yticklabels, ylabel=None):
    xixs = np.arange(1, len(keys) + 1)
    for xix, key in zip(xixs, keys):
        values = results[key][feature]
        hist_counts, hist_bins = np.histogram(values, bins=20)
        fpl.scatter_box_violin_distribution(ax, xix, copy.copy(values), hist_counts, hist_bins,
                                            color=COLORS[key], markersize=MARKERSIZE)
    ax.set_xlim(xixs[0]-0.5, xixs[-1]+0.5)
    ax.set_ylim(*ylim)
    ax.set_xticks(xixs)
    ax.set_yticks(yticks)
    ax.tick_params(axis='y', pad=2)
    ax.tick_params(axis='x', pad=2)
    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'],
                                     tick_length=2.5,
                                     spine_locations={'left': 5, 'bottom': 5},
                                     linewidth=0.5)
    # tick labels after adjust_spines, which would otherwise reset them
    set_scenario_xticks(ax, keys)
    ax.set_yticklabels(yticklabels)
    if ylabel is not None:
        ax.set_ylabel(ylabel, labelpad=-2)
    fifi.mpl_functions.set_fontsize(ax, 6)
    set_selective_rasterization(ax, rasterize_markers=['o'])


# slopes: low-range axis (same 0-8 rad/s as the fly figures) and high-range axis
ax = layout.axes[(fifi_figure_label, 'slopes')]
plot_feature(ax, LOW_SCENARIOS, 'slopes', SLOPE_YLIM, [0, 2, 4, 6, 8], ['0', '', '', '', '8'], 'Slope (rad/s)')

ax = layout.axes[(fifi_figure_label, 'slopes_high')]
plot_feature(ax, HIGH_SCENARIOS, 'slopes', SLOPE_HIGH_YLIM, [0, 10, 20, 30], ['0', '', '', '30'])

# axis ratios: all scenarios on one axis
ax = layout.axes[(fifi_figure_label, 'axis_ratios')]
plot_feature(ax, SCENARIOS, 'axis_ratios', (0, 1), [0, 0.2, 0.4, 0.6, 0.8, 1.0], ['0', '', '', '', '', '1'], 'Axis ratio')

In [9]:
save_layer(layout, fifi_figure_label)